# Supplementary - Mode variograms

Related to Supplementary Figure S6.

In [ ]:
import sys
sys.path.append("..")

from main import *
from visualization import *

In [ ]:
vertices = np.load('../Files/vertices_ellipse.npy').astype('float')
eigenmodes = np.load('../Files/eigenmodes_ellipse.npy')

ellipse = Geometry(vertices, eigenmodes)

# Computing eigenmode variograms/wavelengths

In [ ]:
from scipy.signal import find_peaks
from scipy.stats import zscore

def compute_pairwise_distances(points):
    diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff**2, axis=-1))
    return distances

def compute_variogram(coordinates, values, bins=np.linspace(0, 1, 30), subsample=1000, iters=10):
    variograms = []
    for _ in range(iters):
        random_ids = np.arange(coordinates.shape[0])
        np.random.shuffle(random_ids)
        random_ids = np.sort(random_ids[:subsample])
        d = compute_pairwise_distances(coordinates[random_ids])
        sub_values = values[random_ids]
        variances = (sub_values[np.newaxis, :] - sub_values[:, np.newaxis]) ** 2
        variogram = np.zeros((len(bins) - 1, ))
        for i in range(len(variogram)):
            variogram[i] = np.mean(variances[(d >= bins[i]) & (d < bins[i + 1])])
        variograms.append(variogram)
    return np.mean(np.stack(variograms, axis=0), axis=0)

def get_wavelength(variogram, bins):
    result = find_peaks(variogram)
    if any(result[0]):
        wavelength = 2 * (bins[find_peaks(variogram)[0][0]] + ((bins[1] - bins[0]) / 2))
        return wavelength
    else:
        return 2 * bins[-1]

In [ ]:
bins = np.linspace(0, 1, 61, endpoint=True)

variogram = compute_variogram(vertices, zscore(eigenmodes[:, 1]), bins=bins, subsample=2500)

In [ ]:
bins = np.linspace(0, 1, 61, endpoint=True)

wavelengths, variograms = [], []
for i in tqdm(range(eigenmodes.shape[1])):
    variogram = compute_variogram(vertices, zscore(eigenmodes[:, i]), bins=bins, subsample=2500)
    variograms.append(variogram)
    wavelengths.append(get_wavelength(variogram, bins))

In [ ]:
plt.plot(wavelengths)

In [ ]:
np.save('../Results/mode_variograms2.npy', variograms)
np.save('../Results/mode_wavelengths2.npy', wavelengths)

# Loading data

Variograms were computed prior in the `Figure3-Analysis.ipynb` notebook.

In [ ]:
variograms = np.load('../Results/mode_variograms.npy') 
wavelengths = np.load('../Results/mode_wavelengths.npy')

In [ ]:
vertices = np.load('../Files/vertices_ellipse.npy').astype('float')
eigenmodes = np.load('../Files/eigenmodes_ellipse.npy')

geometry = Geometry(vertices, eigenmodes)
geometry.vertices -= np.mean(geometry.vertices, axis=0)
vertices = geometry.vertices
eigenmodes = geometry.eigenmodes
#vertices = geometry.vertices

Generating RGB arrays of the 3D scatter plots for easier handling in the multipanel figure below.

In [ ]:
figs_ellipse_eigenmodes = []
for i in range(10):
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(5, 5), dpi=300)
    ax.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2], c=eigenmodes[i+1], alpha=0.5, cmap='coolwarm', edgecolor='None')
    ax.set_xlim([-0.5, 0.5])
    ax.set_ylim([-0.5, 0.5])
    ax.set_zlim([-0.5, 0.5])
    ax.set_axis_off()
    plt.tight_layout(pad=0)
    figs_ellipse_eigenmodes.append(figure_to_array(fig))
    plt.close()

In [ ]:
plt.imshow(figs_ellipse_eigenmodes[0])

# Generating supplementary figure

In [ ]:
def zoom_crop(array, factor=2, x_offset=0, y_offset=0):
    if factor != 1:
        L = array.shape[0]
        delta = int((1 - (1 / factor)) * L / 2)
        return array[delta+y_offset:-delta+y_offset, delta+x_offset:-delta+x_offset, :]
    else:
        return array

In [ ]:
x = np.linspace(0, 1, 60, endpoint=True)
x = x[:-1] + (x[1] - x[0]) / 2

In [ ]:
fig = PaperFigure(figsize=(7, 5))

fig.set_tick_length(2)
fig.set_font_size(6)
fig.add_background()

w = 1
pad = (7 - 5 * w) / 4
for i in range(5):
    fig.add_axes('mode{}'.format(i), (i * (w + pad), 0), w, w)
    fig.add_axes('variogram{}'.format(i), (i * (w + pad), w), w, 0.6 * w)
    fig.add_axes('mode{}'.format(i+5), (i * (w + pad), 2*w + pad/2), w, w)
    fig.add_axes('variogram{}'.format(i+5), (i * (w + pad), 3*w + pad/2), w, 0.6 * w)

fig.set_line_thickness(0.6)

# --------------------------------------------------------------------------------------------------


for i in range(10):

    ax = fig.axes[f'mode{i}']
    ax.imshow(zoom_crop(figs_ellipse_eigenmodes[i], factor=1.75))
    ax.axis('off')

    ax = fig.axes[f'variogram{i}']
    #ax.plot(x, variograms[i+1] / np.nanmax(variograms[i+1]), color='black', linewidth=0.75)
    ax.fill_between(x, 0, variograms[i+1] / np.nanmax(variograms[i+1]), color='gray', linewidth=0.75, edgecolor='None')
    ax.axvline(wavelengths[i+1] / 2, color='red', linewidth=0.75)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.set_yticks([0, 1])
    ax.spines[['top', 'right']].set_visible(False)

fig.show()

In [ ]:
fig.save('../Figures/supp_variograms_incomplete.svg')

#### Manually annotating the wavelengths

In [ ]:
print(wavelengths)